# Laya V5 — All-28 Activation-Aware Accuracy-First G128 INT8/INT4

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_1p2_All28_ActivationAware_AccuracyFirst_V5_Colab.ipynb)

**Based on uploaded V4 on Tesla T4:** 112/112 projections quantized (71 INT8, 41 INT4); packed state **445.86 MiB** vs **803.05 MiB** FP16 (44.48% reduction). Teacher agreement: **99.74% DEV**, **98.96% AUDIT (failed 99% target)**, **97.5% official TEST**. Native FP16 materialization achieved 0.975× B1/T512 and 1.000× B4/T512.

**V5 improvements:** four case-disjoint partitions (CAL / SELECT / GUARD / AUDIT), real input-RMS-aware sparse residuals, no newly incorrect teacher decisions on either development set, capped JS/margin erosion, INT4 trial ordering informed (not approved) by the 41 V4 selections, exact HF revision, export/reload parity, and CPU/native-FP16 materialization after packed checkpoint export. Audit/test are never used during search.

SDPA runs attention; native FP16 GEMM runs linear projections. **Low-bit disk storage does not imply low-bit GPU inference.** V5 intentionally prioritizes fidelity over the biggest compression ratio. No model is automatically published.


In [ ]:
#@title 1. Install dependencies (native PyTorch SDPA; no custom Triton requirement)
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4" "datasets>=2.20,<5" pandas matplotlib


In [ ]:
#@title 2. Experiment configuration / pinned standalone checkpoint
import os,sys,json,gc,math,random,time,zipfile,copy,importlib.util
from pathlib import Path
from collections import Counter
import torch,numpy as np,pandas as pd
from torch import nn
from torch.nn import functional as F
from huggingface_hub import snapshot_download,HfApi
from IPython.display import display
from urllib.request import urlopen
from safetensors.torch import save_file,load_file
SOURCE_REPO="vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
TRAIN_CASES=1600 #@param {type:"integer"}
CAL_DECISIONS=96 #@param {type:"integer"}
SELECT_DECISIONS=256 #@param {type:"integer"}
GUARD_DECISIONS=128 #@param {type:"integer"}
AUDIT_DECISIONS=384 #@param {type:"integer"}
TEST_TAIL_DECISIONS=400 #@param {type:"integer"}
MAX_LEN=512 #@param {type:"integer"}
BATCH_SIZE=4 #@param {type:"integer"}
INT8_SPARSE_K=16 #@param {type:"integer"}
INT4_SPARSE_K=24 #@param {type:"integer"}
INT4_EXTRA_SPARSE_K=48 #@param {type:"integer"}
MAX_INT4_TRIALS=64 #@param {type:"integer"}
TRY_WEIGHTED_CORRECTIONS=True #@param {type:"boolean"}
MAX_EXTRA_SELECT_JS=0.00005 #@param {type:"number"}
MAX_EXTRA_GUARD_JS=0.00006 #@param {type:"number"}
MAX_EXTRA_EROSION=0.04 #@param {type:"number"}
BENCH_REPEATS=10 #@param {type:"integer"}
SAVE_PACKED_DELTA=True #@param {type:"boolean"}
VERIFY_PACKED_RELOAD=True #@param {type:"boolean"}
SEED=2051
OUT=Path("/content/Laya_V5_AccuracyFirst_All28")
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Select T4/L4 GPU"
assert torch.cuda.get_device_capability()[0]>=7
try:
    from google.colab import userdata
    TOKEN=userdata.get("HF_TOKEN")
except Exception:TOKEN=os.environ.get("HF_TOKEN")
SOURCE_REV=HfApi(token=TOKEN).model_info(SOURCE_REPO).sha
ROOT=Path(snapshot_download(repo_id=SOURCE_REPO,revision=SOURCE_REV,token=TOKEN))
META=json.loads((ROOT/"config.json").read_text())
assert META.get("format")=="laya-1p2-full-standalone-v1"
assert META.get("dtype_on_disk")=="float16"
print("CUDA",torch.cuda.get_device_name(0),"checkpoint SHA",SOURCE_REV)


In [ ]:
#@title 3. Load original Laya with native PyTorch SDPA
def import_module(path,name):
    spec=importlib.util.spec_from_file_location(name,str(path))
    if spec is None or spec.loader is None:raise ImportError(path)
    m=importlib.util.module_from_spec(spec);sys.modules[name]=m
    spec.loader.exec_module(m);return m
source=import_module(ROOT/"modeling_laya_standalone.py","laya_v5_source")
teacher,tokenizer=source.load_model(ROOT,device="cpu",local_files_only=True,compile_head=False)
teacher=teacher.to("cuda",dtype=torch.float16).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
try:
    if hasattr(teacher.encoder,"set_attn_implementation"):
        teacher.encoder.set_attn_implementation("sdpa")
    else:teacher.encoder.config._attn_implementation="sdpa"
except Exception as exc:print("Attention selection advisory:",str(exc)[:150])
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
assert len(teacher.encoder.layers)==28
print("FP16 source parameters",sum(p.numel() for p in teacher.parameters()))


In [ ]:
#@title 4. Pinned codec, activation-aware runtime, V4 prior; numerical GPU self-tests
RAW="https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/"
CODEC_REV="26a994aa45e838a90fd5b37a6be3b73efc972085"
V5_REV="8badb491e5611c37f4de3aee71aca48f50b94cd9"
PRIOR_REV="c0e03478f87ac715b3ec85982101dafa4d7f26eb"
def retrieve(revision,path):
    dest=Path("/content")/Path(path).name
    with urlopen(f"{RAW}{revision}/{path}",timeout=90) as response:
        dest.write_bytes(response.read())
    assert dest.stat().st_size>500
    return dest
q=import_module(retrieve(CODEC_REV,"src/tinycenn_lm/laya_mixedbit_sparse_v4.py"),
                "laya_mixedbit_sparse_v4")
q5=import_module(retrieve(V5_REV,"src/tinycenn_lm/laya_mixedbit_accuracy_v5.py"),
                 "laya_mixedbit_accuracy_v5")
PRIOR=json.loads(retrieve(PRIOR_REV,"configs/laya_v4_verified_trial_prior.json").read_text())
PRIOR_NAMES=(set(PRIOR["accepted_int4_names"])
             if PRIOR["source_revision"]==SOURCE_REV else set())
for bits in (2,4,8):
    linear=nn.Linear(193,97,device="cuda",dtype=torch.float16)
    sample=torch.randn(9,193,device="cuda",dtype=torch.float16)
    stats={"rms":torch.rand(193)+0.6,"mean":torch.zeros(193)}
    for style in ("plain","weighted"):
        module=q5.quantize_aware(linear,bits,16,stats,style)
        assert torch.equal(q.pack_codes(q.unpack_codes(
            module.packed_weight,bits,module.padded),bits),module.packed_weight)
        with torch.inference_mode():
            dense=F.linear(sample,module.weight_fp16(),module.bias)
            wrapped=module(sample)
            native=module.to_native_fp16()(sample)
        torch.testing.assert_close(dense,wrapped,atol=0.02,rtol=0.02)
        torch.testing.assert_close(dense,native,atol=0.02,rtol=0.02)
    print(bits,"bit weighted/plain GPU parity PASS")
    del linear,sample,module
gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 5. Four strictly SOURCE-CASE-disjoint train partitions
from datasets import load_dataset
def as_obj(v):
    if isinstance(v,str):
        try:return json.loads(v)
        except Exception:return v
    return v
def parse_items(rows,target):
    groups=[]
    for row in rows:
        state=as_obj(row.get("state"));questions=as_obj(row.get("questions"))
        if not isinstance(questions,dict):continue
        case=[]
        for raw in questions.values():
            try:
                question=source.normalize_question(as_obj(raw))
                ids,markers=source.build_sequence(tokenizer,state,question,
                    max_len=MAX_LEN,head_max_len=192,truncate_left=isinstance(state,list))
                if len(markers)<2:continue
                case.append(dict(ids=ids,markers=markers,qtype=source.QTYPES[question["t"]]))
            except (TypeError,ValueError,KeyError,IndexError):continue
        if case:groups.append(case)
    out=[]
    for depth in range(max(map(len,groups),default=0)):
        for group in groups:
            if depth<len(group):
                out.append(group[depth])
                if len(out)>=target:return out
    return out
def batchify(items,size=BATCH_SIZE,pad_len=None):
    batches=[]
    for idx in range(0,len(items),size):
        b=source.collate_items(items[idx:idx+size],int(tokenizer.pad_token_id))
        if pad_len is not None:
            remain=pad_len-b["input_ids"].shape[1]
            if remain<0:raise ValueError("Benchmark padding smaller than example")
            b["input_ids"]=F.pad(b["input_ids"],(0,remain),value=int(tokenizer.pad_token_id))
            b["attention_mask"]=F.pad(b["attention_mask"],(0,remain),value=0)
        batches.append({k:v.to("cuda") for k,v in b.items()})
    return batches
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split=f"train[:{TRAIN_CASES}]"))
assert len(rows)>=1000
random.Random(SEED).shuffle(rows)
cuts=[0,int(len(rows)*.20),int(len(rows)*.50),int(len(rows)*.70),len(rows)]
limits=[CAL_DECISIONS,SELECT_DECISIONS,GUARD_DECISIONS,AUDIT_DECISIONS]
CAL_ITEMS,SEL_ITEMS,GUARD_ITEMS,AUDIT_ITEMS=[
    parse_items(rows[cuts[i]:cuts[i+1]],limits[i]) for i in range(4)]
assert [len(CAL_ITEMS),len(SEL_ITEMS),len(GUARD_ITEMS),len(AUDIT_ITEMS)]==limits,    "Not enough source decisions; reduce limits or increase TRAIN_CASES"
CAL,SEL,GUARD,AUDIT=[batchify(items) for items in
                     (CAL_ITEMS,SEL_ITEMS,GUARD_ITEMS,AUDIT_ITEMS)]
print("CAL/SEL/GUARD/AUDIT:",limits)


In [ ]:
#@title 6. Teacher caches, real activation calibration, zero-new-error objective
@torch.inference_mode()
def infer(model,b):
    with torch.autocast("cuda",dtype=torch.float16):
        logits=model(**b)
    return logits[0] if isinstance(logits,(tuple,list)) else logits
@torch.inference_mode()
def teacher_outputs(batches):
    return [infer(teacher,b).detach().float().cpu() for b in batches]
REF_SEL=teacher_outputs(SEL)
REF_GUARD=teacher_outputs(GUARD)
REF_AUDIT=teacher_outputs(AUDIT)
NAMES=[f"encoder.layers.{layer}.{block}.{component}" for layer in range(28)
       for block,component in (("attn","Wqkv"),("attn","Wo"),
                               ("mlp","Wi"),("mlp","Wo"))]
assert len(NAMES)==112
MOMENTS=q5.capture_input_moments(teacher,NAMES,CAL,stride=8,max_batches=12)
assert all(v["count"]>0 for v in MOMENTS.values())
print("Activation moments collected for",len(MOMENTS),"projections")


In [ ]:
#@title 7. Force 112/112 projections to INT8 + top16 residual; measure both DEV pools
student=copy.deepcopy(teacher).eval().requires_grad_(False)
SOURCE_BYTES=q.state_bytes(teacher)
PRECISION={};SPARSE={};STYLE={}
for layer in range(28):
    for name in NAMES[4*layer:4*layer+4]:
        dense=student.get_submodule(name)
        assert isinstance(dense,nn.Linear)
        module=q5.quantize_aware(dense,8,INT8_SPARSE_K,
                                 stats=MOMENTS[name],style="plain")
        q.swap(student,name,module)
        PRECISION[name]=8;SPARSE[name]=INT8_SPARSE_K;STYLE[name]="plain"
    print("INT8 forced layer",layer)
assert len(PRECISION)==112
BASE_SEL=q5.collect_decisions(student,SEL,REF_SEL)
BASE_GUARD=q5.collect_decisions(student,GUARD,REF_GUARD)
CUR_SEL,CUR_GUARD=BASE_SEL,BASE_GUARD
def score():
    return (q5.collect_decisions(student,SEL,REF_SEL),
            q5.collect_decisions(student,GUARD,REF_GUARD))
def acceptable(new_sel,new_guard,old_sel,old_guard):
    return (q5.safe_accept(new_sel,BASE_SEL,old_sel,
                   BASE_SEL["mean_js"]+MAX_EXTRA_SELECT_JS,allowed_new=0,
                   max_extra_erosion=MAX_EXTRA_EROSION)
            and q5.safe_accept(new_guard,BASE_GUARD,old_guard,
                   BASE_GUARD["mean_js"]+MAX_EXTRA_GUARD_JS,allowed_new=0,
                   max_extra_erosion=MAX_EXTRA_EROSION))
print("INT8 selection baseline",q5.public_metrics(BASE_SEL))
print("INT8 guard baseline",q5.public_metrics(BASE_GUARD))


In [ ]:
#@title 8. Accuracy-first INT4 search with dual development decision-mask guards
# 41 V4 INT4 names guide trial order ONLY. No V4 audit/test labels or
# metrics influence acceptance. Both pools below are TRAIN subsets.
order=sorted(NAMES,key=lambda n:(
    0 if n in PRIOR_NAMES else 1,-int(n.split(".")[2]),n.rsplit(".",1)[-1]))
s_sel=SEL[:min(len(SEL),math.ceil(64/BATCH_SIZE))]
s_guard=GUARD[:min(len(GUARD),math.ceil(32/BATCH_SIZE))]
sr_sel=REF_SEL[:len(s_sel)];sr_guard=REF_GUARD[:len(s_guard)]
attempts=[]
for trial,name in enumerate(order[:MAX_INT4_TRIALS],1):
    dense=teacher.get_submodule(name)
    original=student.get_submodule(name)
    modes=[("plain",INT4_SPARSE_K)]
    if TRY_WEIGHTED_CORRECTIONS:
        modes += [("weighted",INT4_SPARSE_K),
                  ("weighted",INT4_EXTRA_SPARSE_K)]
    for style,k in modes:
        candidate=q5.quantize_aware(dense,4,k,stats=MOMENTS[name],style=style)
        saving=q.state_bytes(original)-q.state_bytes(candidate)
        if saving<=0:continue
        q.swap(student,name,candidate)
        try:
            fast_s=q5.collect_decisions(student,s_sel,sr_sel)
            fast_g=q5.collect_decisions(student,s_guard,sr_guard)
            early_ok=(not any(x and not b for x,b in zip(
                            fast_s["error_mask"],CUR_SEL["error_mask"][:fast_s["n"]]))
                      and not any(x and not b for x,b in zip(
                            fast_g["error_mask"],CUR_GUARD["error_mask"][:fast_g["n"]]))
                      and fast_s["mean_js"]<0.0006
                      and fast_g["mean_js"]<0.0006)
            if early_ok:
                new_sel,new_guard=score()
                keep=acceptable(new_sel,new_guard,CUR_SEL,CUR_GUARD)
            else:keep=False;new_sel=new_guard=None
        except Exception as exc:
            keep=False;new_sel=new_guard=None
            print("Rejected error",name,style,k,repr(exc)[:170])
        if keep:
            CUR_SEL,CUR_GUARD=new_sel,new_guard
            PRECISION[name]=4;SPARSE[name]=k;STYLE[name]=style
            print(f"ACCEPT {name}: INT4 {style} sparse{k}, "
                  f"select={new_sel['agreement']:.3%},guard={new_guard['agreement']:.3%}")
        else:q.swap(student,name,original)
        attempts.append(dict(module=name,mode=style,sparse_k=k,
          saved_mib=saving/2**20,accepted=keep,
          select_agree=None if new_sel is None else new_sel["agreement"],
          guard_agree=None if new_guard is None else new_guard["agreement"]))
        if keep:break
    if trial%4==0:
        pd.DataFrame(attempts).to_csv(OUT/"int4_trials_v5.csv",index=False)
        (OUT/"chosen_precision_v5.json").write_text(json.dumps(PRECISION,indent=2))
        print("Trial",trial,"accepted INT4",sum(v==4 for v in PRECISION.values()),
              "packed MiB",round(q.state_bytes(student)/2**20,2))
FINAL_SEL,FINAL_GUARD=score()
assert acceptable(FINAL_SEL,FINAL_GUARD,CUR_SEL,CUR_GUARD)
assert len(PRECISION)==112 and all(
    isinstance(student.get_submodule(n),q.PackedSparseLinear) for n in NAMES)
print("Selected",q5.public_metrics(FINAL_SEL))
print("Guard",q5.public_metrics(FINAL_GUARD))


In [ ]:
#@title 9. Sealed independent audit and later-test-case diagnostic
FINAL_AUDIT=q5.collect_decisions(student,AUDIT,REF_AUDIT)
AUDIT_PASS=(FINAL_AUDIT["agreement"]>=0.99 and
            FINAL_AUDIT["mean_js"]<=0.0015)
print("SEALED AUDIT",q5.public_metrics(FINAL_AUDIT),
      "PASS" if AUDIT_PASS else "FAIL")
TEST_DIAGNOSTIC={}
if TEST_TAIL_DECISIONS:
    tests=list(load_dataset("LocalLLaMA/typed-decisions","all",split="test"))
    offset=800 if len(tests)>=1300 else int(len(tests)*0.60)
    held=parse_items(tests[offset:],TEST_TAIL_DECISIONS)
    if held:
        bt=batchify(held)
        truth=teacher_outputs(bt)
        metric=q5.collect_decisions(student,bt,truth)
        TEST_DIAGNOSTIC=q5.public_metrics(metric)
        TEST_DIAGNOSTIC.update(case_offset=offset,total_test_cases=len(tests),
            fresh_relative_to_V4_not_guaranteed=len(tests)<1300)
        print("Case-tail TEST diagnostic",TEST_DIAGNOSTIC)
print("Do not tune the model using audit or test results.")


In [ ]:
#@title 10. Export packed delta with strict revision, cold-reload parity
PACKED_BYTES=q.state_bytes(student)
manifest=dict(source_repo=SOURCE_REPO,source_revision=SOURCE_REV,
  codec_revision=CODEC_REV,v5_revision=V5_REV,
  precision_by_module=PRECISION,sparse_by_module=SPARSE,style_by_module=STYLE,
  group_size=128,source_checkpoint_required=True)
(OUT/"manifest_v5.json").write_text(json.dumps(manifest,indent=2))
DELTA=OUT/"laya_v5_all28_sparse_delta.safetensors"
if SAVE_PACKED_DELTA:
    state={}
    for name in NAMES:
        for suffix,v in student.get_submodule(name).state_dict().items():
            state[f"{name}.{suffix}"]=v.detach().cpu().contiguous()
    save_file(state,str(DELTA));del state
    print("Packed delta MiB",DELTA.stat().st_size/2**20)
    if VERIFY_PACKED_RELOAD:
        rebuilt=copy.deepcopy(teacher).eval().requires_grad_(False)
        weights=load_file(str(DELTA),device="cpu")
        q.load_packed_delta_in_place(rebuilt,manifest,weights)
        del weights
        with torch.inference_mode():
            a=infer(student,SEL[0]).float()
            b=infer(rebuilt,SEL[0]).float()
        torch.testing.assert_close(a,b,atol=0.003,rtol=0.003)
        print("Checkpoint reload parity PASS",float((a-b).abs().max()))
        del rebuilt,a,b
        gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 11. GPU benchmark + drop packed buffers to form efficient native FP16 execution
@torch.inference_mode()
def measure(fn):
    for _ in range(4):fn()
    torch.cuda.synchronize()
    durations=[]
    for _ in range(BENCH_REPEATS):
        a=torch.cuda.Event(enable_timing=True)
        b=torch.cuda.Event(enable_timing=True)
        a.record();fn();b.record();b.synchronize()
        durations.append(a.elapsed_time(b))
    return float(np.median(durations))
records=[]
for B in (1,4):
    examples=SEL_ITEMS[:B]
    min_len=max(len(i["ids"]) for i in examples)
    for T in (128,256,512):
        if T<min_len or T>MAX_LEN:continue
        batch=batchify(examples,size=B,pad_len=T)[0]
        original=measure(lambda:infer(teacher,batch))
        packed=measure(lambda:infer(student,batch))
        records.append(dict(batch=B,tokens=T,fp16_ms=original,
                            packed_cached_ms=packed,packed_speedup=original/packed))
print("Packed and FP16 cache resident model MiB",
      q.resident_bytes(student)/2**20)
q.remove_caches(student)
q.materialize_native_in_place(student,NAMES)
assert all(isinstance(student.get_submodule(n),nn.Linear) for n in NAMES)
assert abs(q.state_bytes(student)-q.state_bytes(teacher))<32768
torch.cuda.empty_cache()
for x in records:
    b=batchify(SEL_ITEMS[:x["batch"]],size=x["batch"],pad_len=x["tokens"])[0]
    ms=measure(lambda:infer(student,b))
    x["native_fp16_ms"]=ms
    x["native_speedup"]=x["fp16_ms"]/ms
    x["packed_state_mib"]=PACKED_BYTES/2**20
    x["native_state_mib"]=q.state_bytes(student)/2**20
    print("B",x["batch"],"T",x["tokens"],"speedup",round(x["native_speedup"],3))
LATENCY=pd.DataFrame(records)
assert len(LATENCY)
display(LATENCY)
LATENCY.to_csv(OUT/"latency_v5.csv",index=False)
NATIVE_SEL=q5.collect_decisions(student,SEL,REF_SEL)
assert abs(NATIVE_SEL["agreement"]-FINAL_SEL["agreement"])<=1/SELECT_DECISIONS


In [ ]:
#@title 12. Honest quality, compression and speed summary
report=dict(
  model=SOURCE_REPO,model_revision=SOURCE_REV,
  gpu=torch.cuda.get_device_name(0),
  layers=28,quantized_modules=len(PRECISION),
  precision_distribution=dict(Counter(PRECISION.values())),
  original_state_mib=SOURCE_BYTES/2**20,
  packed_state_mib=PACKED_BYTES/2**20,
  compression_percent=100*(1-PACKED_BYTES/SOURCE_BYTES),
  native_fp16_state_mib=q.state_bytes(student)/2**20,
  original_v4_packed_mib=PRIOR["v4_packed_mib"],
  all8_select=q5.public_metrics(BASE_SEL),
  all8_guard=q5.public_metrics(BASE_GUARD),
  select=q5.public_metrics(FINAL_SEL),
  guard=q5.public_metrics(FINAL_GUARD),
  independent_audit=q5.public_metrics(FINAL_AUDIT),
  audit_pass=bool(AUDIT_PASS),official_test_tail=TEST_DIAGNOSTIC,
  native_select=q5.public_metrics(NATIVE_SEL),
  speedups=LATENCY.native_speedup.tolist(),
  all_speedups_gt_1=bool((LATENCY.native_speedup>1).all()),
  production_validated=False,
  notes=["V4 prior controls test ordering only; no reuse of V4 audit/test decisions",
         "Official test was previously evaluated in V4: tail diagnostic is not a clean independent study",
         "Native FP16 runtime uses original-sized GPU weights; compact delta requires exact HF base",
         "No native low-bit GEMM acceleration; SDPA applies only to attention"])
(OUT/"summary_v5.json").write_text(json.dumps(report,indent=2))
print(json.dumps({k:report[k] for k in (
  "precision_distribution","compression_percent","select","guard",
  "independent_audit","audit_pass","official_test_tail","speedups")},indent=2))


In [ ]:
#@title 13. Visualize precision and speed
import matplotlib.pyplot as plt
n=Counter(PRECISION.values())
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(["INT4","INT8"],[n.get(4,0),n.get(8,0)])
axes[0].set(xlabel="Packed precision",ylabel="Projections",title="All 112 modules")
axes[1].bar([f"B{int(x.batch)} T{int(x.tokens)}" for x in LATENCY.itertuples()],
            LATENCY.native_speedup)
axes[1].axhline(1.0,linestyle="--")
axes[1].set(ylabel="Speedup vs original FP16",title="Native FP16, PyTorch SDPA")
axes[1].tick_params(axis="x",rotation=30)
fig.tight_layout()
fig.savefig(OUT/"v5_speed_and_bits.png",dpi=160)
plt.show()


In [ ]:
#@title 14. Results ZIP (large packed checkpoint delta is separate)
archive=OUT/"Laya_V5_AccuracyFirst_All28_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file() and f not in (DELTA,archive):z.write(f,f.name)
print("Diagnostics:",archive)
if SAVE_PACKED_DELTA:print("Separate packed weights file:",DELTA)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass


## Reading V5

Every one of the 112 encoder projections is quantized to at least INT8. No 4-bit choice is accepted unless it avoids *new teacher-decision errors* on **both** training-development partitions, and passes JS/confidence margin limits. The sealed independent audit and test-tail diagnostics are not used for search.

The new activation-weighted sparse residual corrector uses real projection input RMS collected **only on calibration data**. V4's 41 successful INT4 projections merely define a cost-effective search order.

The exported safetensors file is a **packed delta**, not a standalone model; loading requires the exact base Hugging Face revision. Native FP16 materialization trades away low-bit active VRAM but removes the V4 double-residency problem. The notebook does not publish any model automatically.
